# Final Train-Test Split Pipeline with Window/Trial Accuracy

This notebook implements the complete train-test split pipeline for EEG-based emotion recognition using LDA, including window-level and trial-level accuracy reporting.

## 1. Import Libraries and Set Constants
Import necessary libraries, define constants, and set up paths for data and output files.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import warnings
import itertools
from pathlib import Path
from collections import defaultdict

# ── Numerical / Scientific ────────────────────────────────────────────────────
import numpy as np
import scipy.io as sio
import scipy.signal as signal
from scipy.linalg import eigvalsh

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline

# ── Riemannian Geometry ───────────────────────────────────────────────────────
try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    PYRIEMANN_AVAILABLE = True
except ImportError:
    PYRIEMANN_AVAILABLE = False
    warnings.warn("pyriemann not found. Riemannian features will be skipped.")

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Constants ─────────────────────────────────────────────────────────────────
FS_EEG   = 128          # EEG sampling frequency (Hz)
FS_BVP   = 64           # BVP sampling frequency (Hz)
N_TRIALS = 40           # Number of trials per subject
N_CLASSES = 2           # Binary classification (e.g., high/low valence)
BASELINE_DURATION = 3   # seconds
TRIAL_DURATION    = 60  # seconds
WINDOW_SIZE       = 4   # seconds
STEP_SIZE         = 2   # seconds (50 % overlap)
RANDOM_STATE      = 42

# ── Frequency Bands ───────────────────────────────────────────────────────────
BANDS = {
    "delta": (1,   4),
    "theta": (4,   8),
    "alpha": (8,  13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}

# ── FAA Electrode Pairs (F3/F4 → indices for DEAP) ───────────────────────────
FAA_PAIRS = [(0, 16)]   # (left-frontal idx, right-frontal idx)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"e:\FInal Year Project\LDACode")
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✔  Libraries imported.")
print(f"   Data  dir : {DATA_DIR}")
print(f"   Output dir: {OUTPUT_DIR}")

## 2. Load and Preprocess Data
Load training data, apply baseline reduction, and preprocess signals for feature extraction.

In [ ]:
def load_deap_subject(filepath: Path) -> dict:
    """Load a single DEAP .mat file and return a dict with data & labels."""
    mat = sio.loadmat(str(filepath))
    return {
        "data":   mat["data"].astype(np.float64),    # (40, 40, 8064)
        "labels": mat["labels"].astype(np.float64),  # (40,  4)
    }


def apply_baseline_reduction(trial_data: np.ndarray,
                              fs: int = FS_EEG,
                              baseline_sec: int = BASELINE_DURATION) -> np.ndarray:
    """Subtract per-channel baseline mean from trial data.

    Parameters
    ----------
    trial_data : np.ndarray  shape (n_channels, n_samples)
    """
    baseline_samples = baseline_sec * fs
    baseline_mean = trial_data[:, :baseline_samples].mean(axis=1, keepdims=True)
    return trial_data[:, baseline_samples:] - baseline_mean


def bandpass_filter(data: np.ndarray, low: float, high: float,
                    fs: int = FS_EEG, order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass filter applied per channel."""
    nyq  = fs / 2.0
    b, a = signal.butter(order, [low / nyq, high / nyq], btype="band")
    return signal.filtfilt(b, a, data, axis=-1)


def preprocess_subject(subject_data: dict) -> dict:
    """Apply baseline reduction and bandpass (1–45 Hz) to EEG channels."""
    raw    = subject_data["data"]          # (40, 40, 8064)
    labels = subject_data["labels"]        # (40,  4)
    eeg    = raw[:, :32, :]               # first 32 channels → EEG
    periph = raw[:, 32:, :]               # channels 32-39   → peripheral

    processed_eeg  = np.zeros((eeg.shape[0], eeg.shape[1],
                                eeg.shape[2] - BASELINE_DURATION * FS_EEG))
    processed_per  = np.zeros((periph.shape[0], periph.shape[1],
                                periph.shape[2] - BASELINE_DURATION * FS_EEG))

    for trial_idx in range(eeg.shape[0]):
        trial_eeg = apply_baseline_reduction(eeg[trial_idx])
        trial_eeg = bandpass_filter(trial_eeg, 1.0, 45.0)
        processed_eeg[trial_idx] = trial_eeg

        trial_per = apply_baseline_reduction(periph[trial_idx])
        processed_per[trial_idx] = trial_per

    binary_labels = (labels[:, 0] >= 5).astype(int)  # valence ≥ 5 → positive
    return {"eeg": processed_eeg, "peripheral": processed_per,
            "labels": binary_labels, "raw_labels": labels}


# ── Load all subjects ─────────────────────────────────────────────────────────
subject_files = sorted(DATA_DIR.glob("s*.mat"))
print(f"Found {len(subject_files)} subject file(s) in {DATA_DIR}")

all_subjects = {}
for fpath in subject_files:
    sid = fpath.stem          # e.g. "s01"
    print(f"  Loading {sid} …", end=" ")
    raw_data = load_deap_subject(fpath)
    all_subjects[sid] = preprocess_subject(raw_data)
    print("done.")

print(f"\n✔  Loaded and preprocessed {len(all_subjects)} subject(s).")

## 3. Feature Extraction
Extract features from EEG band powers, BVP, HR, PPI, FAA, and Riemannian geometry.

In [ ]:
# ── 3.1  Band-Power Features ──────────────────────────────────────────────────

def compute_band_powers(eeg_window: np.ndarray, fs: int = FS_EEG) -> np.ndarray:
    """Return log band-power for each channel × frequency band.

    Parameters
    ----------
    eeg_window : np.ndarray  shape (n_channels, n_samples)

    Returns
    -------
    features : np.ndarray  shape (n_channels * n_bands,)
    """
    n_channels = eeg_window.shape[0]
    features   = []
    freqs, psd = signal.welch(eeg_window, fs=fs, nperseg=fs * 2, axis=-1)

    for band, (lo, hi) in BANDS.items():
        idx  = np.where((freqs >= lo) & (freqs < hi))[0]
        power = np.log(psd[:, idx].mean(axis=-1) + 1e-12)  # (n_channels,)
        features.append(power)

    return np.concatenate(features)   # (n_channels * n_bands,)


# ── 3.2  Frontal Alpha Asymmetry (FAA) ───────────────────────────────────────

def compute_faa(eeg_window: np.ndarray, fs: int = FS_EEG) -> np.ndarray:
    """Compute FAA = log(alpha_right) - log(alpha_left) for each pair."""
    freqs, psd = signal.welch(eeg_window, fs=fs, nperseg=fs * 2, axis=-1)
    alpha_idx  = np.where((freqs >= 8) & (freqs < 13))[0]
    faa_feats  = []
    for left_idx, right_idx in FAA_PAIRS:
        log_left  = np.log(psd[left_idx,  alpha_idx].mean() + 1e-12)
        log_right = np.log(psd[right_idx, alpha_idx].mean() + 1e-12)
        faa_feats.append(log_right - log_left)
    return np.array(faa_feats)


# ── 3.3  BVP / HR / PPI Features ─────────────────────────────────────────────

def compute_bvp_features(bvp_signal: np.ndarray,
                          fs: int = FS_BVP) -> np.ndarray:
    """Extract statistical and frequency features from BVP."""
    mean_val = np.mean(bvp_signal)
    std_val  = np.std(bvp_signal)
    skew     = float(np.mean(((bvp_signal - mean_val) / (std_val + 1e-9)) ** 3))
    kurt     = float(np.mean(((bvp_signal - mean_val) / (std_val + 1e-9)) ** 4))

    # Peak detection → inter-peak intervals (PPI)
    peaks, _ = signal.find_peaks(bvp_signal, distance=int(fs * 0.4))
    if len(peaks) > 1:
        ppi      = np.diff(peaks) / fs          # seconds
        hr_est   = 60.0 / ppi.mean()
        ppi_std  = ppi.std()
        rmssd    = np.sqrt(np.mean(np.diff(ppi) ** 2))
    else:
        hr_est  = 0.0
        ppi_std = 0.0
        rmssd   = 0.0

    return np.array([mean_val, std_val, skew, kurt, hr_est, ppi_std, rmssd])


# ── 3.4  Riemannian Geometry Features ────────────────────────────────────────

def compute_riemannian_features(eeg_window: np.ndarray) -> np.ndarray:
    """Project covariance matrix onto tangent space."""
    if not PYRIEMANN_AVAILABLE:
        return np.array([])
    X = eeg_window[np.newaxis, :, :]          # (1, C, T)
    cov = Covariances(estimator="lwf").fit_transform(X)
    ts  = TangentSpace(metric="riemann")
    ts.fit(cov)
    return ts.transform(cov).flatten()


# ── 3.5  Full Feature Vector for One Window ───────────────────────────────────

def extract_features_window(eeg_window: np.ndarray,
                             bvp_window: np.ndarray) -> np.ndarray:
    """Concatenate all feature groups for a single window."""
    bp_feats   = compute_band_powers(eeg_window)
    faa_feats  = compute_faa(eeg_window)
    bvp_feats  = compute_bvp_features(bvp_window)
    riem_feats = compute_riemannian_features(eeg_window)

    parts = [bp_feats, faa_feats, bvp_feats]
    if len(riem_feats):
        parts.append(riem_feats)

    return np.concatenate(parts)


print("✔  Feature extraction functions defined.")
print(f"   Band-power dims  : 32 channels × {len(BANDS)} bands = {32 * len(BANDS)}")
print(f"   FAA dims         : {len(FAA_PAIRS)}")
print(f"   BVP/HR/PPI dims  : 7")
print(f"   Riemannian dims  : {'enabled' if PYRIEMANN_AVAILABLE else 'disabled'}")

In [ ]:
# ── 3.6  Extract Features for All Subjects ────────────────────────────────────

def extract_all_features(subjects: dict,
                          win_sec:  int = WINDOW_SIZE,
                          step_sec: int = STEP_SIZE,
                          fs_eeg:   int = FS_EEG,
                          fs_bvp:   int = FS_BVP) -> tuple:
    """
    Returns
    -------
    X_all      : np.ndarray  (total_windows, n_features)
    y_all      : np.ndarray  (total_windows,)
    meta_all   : list of dicts with subject / trial / window index
    """
    win_eeg  = win_sec  * fs_eeg
    step_eeg = step_sec * fs_eeg
    win_bvp  = win_sec  * fs_bvp
    step_bvp = step_sec * fs_bvp

    # BVP channel index in peripheral block (DEAP: ch 38 = BVP → idx 6)
    BVP_IDX = 6

    X_list, y_list, meta_list = [], [], []

    for sid, subj in subjects.items():
        eeg   = subj["eeg"]          # (40, 32, n_samples)
        periph = subj["peripheral"]  # (40,  8, n_samples)
        labels = subj["labels"]      # (40,)

        for trial_idx in range(eeg.shape[0]):
            trial_eeg = eeg[trial_idx]              # (32, n_samples)
            trial_bvp = periph[trial_idx, BVP_IDX]  # (n_samples,)
            label     = labels[trial_idx]
            n_samples = trial_eeg.shape[1]

            w_start = 0
            win_num = 0
            while w_start + win_eeg <= n_samples:
                eeg_win = trial_eeg[:, w_start : w_start + win_eeg]
                b_start = int(w_start * fs_bvp / fs_eeg)
                bvp_win = trial_bvp[b_start : b_start + win_bvp]

                if len(bvp_win) < win_bvp:
                    bvp_win = np.pad(bvp_win, (0, win_bvp - len(bvp_win)))

                feats = extract_features_window(eeg_win, bvp_win)
                X_list.append(feats)
                y_list.append(label)
                meta_list.append({"subject": sid,
                                   "trial":   trial_idx,
                                   "window":  win_num})
                w_start += step_eeg
                win_num += 1

        print(f"  {sid}: {win_num} windows/trial extracted")

    X_all = np.vstack(X_list)
    y_all = np.array(y_list)
    return X_all, y_all, meta_list


if all_subjects:
    print("Extracting features …")
    X_all, y_all, meta_all = extract_all_features(all_subjects)
    print(f"\n✔  Feature matrix shape : {X_all.shape}")
    print(f"   Label distribution   : {np.bincount(y_all)}")
else:
    print("⚠  No subject data loaded — using synthetic data for demonstration.")
    N_DEMO = 800
    X_all  = np.random.randn(N_DEMO, 32 * 5 + 1 + 7)
    y_all  = np.random.randint(0, 2, N_DEMO)
    meta_all = [{"subject": f"s{i//200+1:02d}",
                 "trial":   (i % 200) // 5,
                 "window":   i % 5} for i in range(N_DEMO)]
    print(f"   Demo matrix shape    : {X_all.shape}")

## 4. Windowing and Augmentation
Divide data into overlapping windows and apply data augmentation techniques like noise addition and mixup.

In [ ]:
# ── 4.1  Gaussian Noise Augmentation ─────────────────────────────────────────

def augment_noise(X: np.ndarray, y: np.ndarray,
                  noise_std: float = 0.05,
                  n_copies: int = 1,
                  random_state: int = RANDOM_STATE) -> tuple:
    """Add Gaussian noise to every sample `n_copies` times."""
    rng = np.random.default_rng(random_state)
    X_aug_list, y_aug_list = [X], [y]
    for _ in range(n_copies):
        noise = rng.normal(0, noise_std, X.shape)
        X_aug_list.append(X + noise)
        y_aug_list.append(y.copy())
    return np.vstack(X_aug_list), np.concatenate(y_aug_list)


# ── 4.2  Mixup Augmentation ───────────────────────────────────────────────────

def augment_mixup(X: np.ndarray, y: np.ndarray,
                  alpha: float = 0.2,
                  n_samples: int = 200,
                  random_state: int = RANDOM_STATE) -> tuple:
    """Generate `n_samples` mixup samples (label-preserving majority vote)."""
    rng    = np.random.default_rng(random_state)
    idx_a  = rng.integers(0, len(X), n_samples)
    idx_b  = rng.integers(0, len(X), n_samples)
    lam    = rng.beta(alpha, alpha, n_samples)[:, np.newaxis]
    X_mix  = lam * X[idx_a] + (1 - lam) * X[idx_b]
    y_mix  = np.where(lam[:, 0] >= 0.5, y[idx_a], y[idx_b])
    return (np.vstack([X, X_mix]),
            np.concatenate([y, y_mix]))


# ── 4.3  Apply Augmentation ───────────────────────────────────────────────────

print("Applying data augmentation …")
print(f"  Before augmentation : {X_all.shape[0]} windows")

X_aug, y_aug = augment_noise(X_all, y_all, noise_std=0.03, n_copies=1)
X_aug, y_aug = augment_mixup(X_aug, y_aug, alpha=0.2, n_samples=300)

print(f"  After augmentation  : {X_aug.shape[0]} windows")
print(f"  Label distribution  : {np.bincount(y_aug)}")

## 5. Train-Test Split and Preprocessing
Assign balanced folds, preprocess features using variance thresholding and z-score normalization.

In [ ]:
from sklearn.model_selection import train_test_split

# ── 5.1  Stratified Train / Test Split ───────────────────────────────────────

TEST_SIZE    = 0.20
N_CV_FOLDS   = 5

# Use original (non-augmented) windows for the held-out test set
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE
)

print(f"Train windows (raw)  : {X_train_raw.shape[0]}")
print(f"Test  windows        : {X_test.shape[0]}")

# Augment only the training partition
X_train_aug, y_train_aug = augment_noise(X_train_raw, y_train_raw,
                                          noise_std=0.03, n_copies=1)
X_train_aug, y_train_aug = augment_mixup(X_train_aug, y_train_aug,
                                          n_samples=200)
print(f"Train windows (aug)  : {X_train_aug.shape[0]}")


# ── 5.2  Variance Threshold ───────────────────────────────────────────────────

var_selector = VarianceThreshold(threshold=1e-6)
X_train_vt   = var_selector.fit_transform(X_train_aug)
X_test_vt    = var_selector.transform(X_test)

n_removed = X_train_aug.shape[1] - X_train_vt.shape[1]
print(f"\nVariance thresholding : removed {n_removed} zero-variance features")
print(f"Remaining features    : {X_train_vt.shape[1]}")


# ── 5.3  Z-Score Normalisation ────────────────────────────────────────────────

scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train_vt)
X_test_sc    = scaler.transform(X_test_vt)

print(f"\nZ-score normalisation : mean ≈ {X_train_sc.mean():.4f}, "
      f"std ≈ {X_train_sc.std():.4f}")
print("✔  Preprocessing complete.")

## 6. Model Training and Validation
Select hyperparameters for LDA, train the model on non-test folds, and validate using cross-validation.

In [ ]:
# ── 6.1  Hyperparameter Grid for LDA ─────────────────────────────────────────

lda_configs = [
    {"solver": "svd",  "shrinkage": None},
    {"solver": "lsqr", "shrinkage": "auto"},
    {"solver": "eigen","shrinkage": "auto"},
    {"solver": "lsqr", "shrinkage": 0.1},
    {"solver": "lsqr", "shrinkage": 0.5},
]

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True,
                     random_state=RANDOM_STATE)

print(f"{'Config':<40} {'Mean CV Acc':>12} {'Std':>8}")
print("-" * 62)

best_score  = -1.0
best_config = None
cv_results  = []

for cfg in lda_configs:
    try:
        lda = LinearDiscriminantAnalysis(**cfg)
        scores = cross_val_score(lda, X_train_sc, y_train_aug,
                                 cv=cv, scoring="accuracy", n_jobs=-1)
        mean_acc, std_acc = scores.mean(), scores.std()
        label = f"solver={cfg['solver']}, shrink={cfg['shrinkage']}"
        print(f"  {label:<38} {mean_acc:>12.4f} {std_acc:>8.4f}")
        cv_results.append({"config": cfg, "mean": mean_acc, "std": std_acc})
        if mean_acc > best_score:
            best_score  = mean_acc
            best_config = cfg
    except Exception as e:
        print(f"  {cfg} → FAILED ({e})")

print(f"\n✔  Best config : {best_config}  →  CV acc = {best_score:.4f}")

In [ ]:
# ── 6.2  Train Final Model on Full Training Set ───────────────────────────────

lda_model = LinearDiscriminantAnalysis(**best_config)
lda_model.fit(X_train_sc, y_train_aug)
print("✔  LDA model trained on full augmented training set.")

# ── 6.3  Cross-Validation Accuracy Curve ─────────────────────────────────────

fold_accs = cross_val_score(lda_model, X_train_sc, y_train_aug,
                             cv=cv, scoring="accuracy")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, N_CV_FOLDS + 1), fold_accs, color="steelblue", alpha=0.8)
ax.axhline(fold_accs.mean(), color="red", linestyle="--",
           label=f"Mean = {fold_accs.mean():.3f}")
ax.set_xlabel("Fold")
ax.set_ylabel("Accuracy")
ax.set_title("Cross-Validation Fold Accuracies")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cv_fold_accuracies.png", dpi=150)
plt.show()
print(f"CV Mean Accuracy : {fold_accs.mean():.4f} ± {fold_accs.std():.4f}")

## 7. Evaluation on Test Split
Evaluate the trained model on the held-out test split and report window-level and trial-level accuracy.

In [ ]:
# ── 7.1  Window-Level Accuracy ────────────────────────────────────────────────

y_pred_windows = lda_model.predict(X_test_sc)
window_acc     = accuracy_score(y_test, y_pred_windows)

print("=" * 55)
print("  WINDOW-LEVEL EVALUATION ON HELD-OUT TEST SET")
print("=" * 55)
print(f"  Accuracy : {window_acc:.4f}  ({window_acc*100:.2f} %)")
print()
print(classification_report(y_test, y_pred_windows,
                             target_names=["Low Valence", "High Valence"]))


# ── 7.2  Confusion Matrix ─────────────────────────────────────────────────────

cm = confusion_matrix(y_test, y_pred_windows)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "High"],
            yticklabels=["Low", "High"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix – Window Level (Test Set)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_window.png", dpi=150)
plt.show()

In [ ]:
# ── 7.3  Trial-Level Accuracy (Majority Vote) ─────────────────────────────────

def trial_level_accuracy(y_true: np.ndarray,
                          y_pred: np.ndarray,
                          meta:   list,
                          split_indices: np.ndarray) -> float:
    """
    Majority-vote aggregation across windows belonging to the same trial.

    Parameters
    ----------
    split_indices : indices into meta_all that correspond to the test split.
    """
    trial_votes   = defaultdict(list)
    trial_gt      = {}

    for local_i, global_i in enumerate(split_indices):
        m   = meta[global_i]
        key = (m["subject"], m["trial"])
        trial_votes[key].append(y_pred[local_i])
        trial_gt[key] = y_true[local_i]

    correct = 0
    total   = len(trial_votes)
    for key, votes in trial_votes.items():
        majority = int(np.round(np.mean(votes)))
        if majority == trial_gt[key]:
            correct += 1

    return correct / total if total > 0 else 0.0


# Recover test indices via reproducible split
indices = np.arange(len(X_all))
_, test_idx = train_test_split(indices,
                                test_size=TEST_SIZE,
                                stratify=y_all,
                                random_state=RANDOM_STATE)

trial_acc = trial_level_accuracy(y_test, y_pred_windows, meta_all, test_idx)

print("=" * 55)
print("  TRIAL-LEVEL EVALUATION (MAJORITY VOTE)")
print("=" * 55)
print(f"  Trial-level Accuracy : {trial_acc:.4f}  ({trial_acc*100:.2f} %)")
print(f"  Window-level Accuracy: {window_acc:.4f}  ({window_acc*100:.2f} %)")
print("=" * 55)

# ── Save results ──────────────────────────────────────────────────────────────
results_summary = {
    "window_accuracy": window_acc,
    "trial_accuracy":  trial_acc,
    "cv_mean":         float(fold_accs.mean()),
    "cv_std":          float(fold_accs.std()),
    "best_lda_config": best_config,
}
import json
with open(OUTPUT_DIR / "test_results.json", "w") as fp:
    json.dump(results_summary, fp, indent=2)
print(f"\n✔  Results saved to {OUTPUT_DIR / 'test_results.json'}")

## 8. LOSO Cross-Subject Evaluation
Perform leave-one-subject-out evaluation to assess cross-subject generalization and report results.

In [ ]:
# ── 8.1  Build per-subject feature arrays ────────────────────────────────────

subj_ids = sorted({m["subject"] for m in meta_all})

subj_X = defaultdict(list)
subj_y = defaultdict(list)

for feat_vec, label, meta in zip(X_all, y_all, meta_all):
    sid = meta["subject"]
    subj_X[sid].append(feat_vec)
    subj_y[sid].append(label)

for sid in subj_ids:
    subj_X[sid] = np.vstack(subj_X[sid])
    subj_y[sid] = np.array(subj_y[sid])

print(f"Subjects available for LOSO : {len(subj_ids)}")
for sid in subj_ids:
    print(f"  {sid} → {subj_X[sid].shape[0]} windows, "
          f"class balance {np.bincount(subj_y[sid])}")

In [ ]:
# ── 8.2  LOSO Loop ────────────────────────────────────────────────────────────

loso_results = {}

for test_sid in subj_ids:
    # Train on all subjects except test_sid
    train_sids = [s for s in subj_ids if s != test_sid]
    X_tr = np.vstack([subj_X[s] for s in train_sids])
    y_tr = np.concatenate([subj_y[s] for s in train_sids])
    X_te = subj_X[test_sid]
    y_te = subj_y[test_sid]

    # Augment training data
    X_tr_a, y_tr_a = augment_noise(X_tr, y_tr, noise_std=0.03, n_copies=1)

    # Preprocess
    vt  = VarianceThreshold(threshold=1e-6).fit(X_tr_a)
    sc  = StandardScaler().fit(vt.transform(X_tr_a))
    X_tr_p = sc.transform(vt.transform(X_tr_a))
    X_te_p = sc.transform(vt.transform(X_te))

    # Train & predict
    clf = LinearDiscriminantAnalysis(**best_config)
    clf.fit(X_tr_p, y_tr_a)
    y_pred = clf.predict(X_te_p)

    win_acc   = accuracy_score(y_te, y_pred)

    # Trial-level majority vote
    trial_votes = defaultdict(list)
    trial_gt    = {}
    for i, m in enumerate([m for m in meta_all if m["subject"] == test_sid]):
        key = m["trial"]
        trial_votes[key].append(y_pred[i])
        trial_gt[key] = y_te[i]

    correct = sum(1 for k, v in trial_votes.items()
                  if int(np.round(np.mean(v))) == trial_gt[k])
    trl_acc = correct / len(trial_votes) if trial_votes else 0.0

    loso_results[test_sid] = {"window_acc": win_acc, "trial_acc": trl_acc}
    print(f"  LOSO {test_sid} → win={win_acc:.4f}  trial={trl_acc:.4f}")

win_loso_mean  = np.mean([v["window_acc"] for v in loso_results.values()])
trl_loso_mean  = np.mean([v["trial_acc"]  for v in loso_results.values()])
print(f"\n✔  LOSO Mean Window Accuracy : {win_loso_mean:.4f}")
print(f"✔  LOSO Mean Trial  Accuracy : {trl_loso_mean:.4f}")

In [ ]:
# ── 8.3  Visualise LOSO Results ───────────────────────────────────────────────

sids       = list(loso_results.keys())
win_accs   = [loso_results[s]["window_acc"] for s in sids]
trl_accs   = [loso_results[s]["trial_acc"]  for s in sids]

x = np.arange(len(sids))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(sids) * 0.7), 5))
bars1 = ax.bar(x - width/2, win_accs, width, label="Window Acc", color="steelblue", alpha=0.85)
bars2 = ax.bar(x + width/2, trl_accs, width, label="Trial Acc",  color="darkorange", alpha=0.85)

ax.axhline(win_loso_mean, color="steelblue",   linestyle="--", alpha=0.6,
           label=f"Mean Win  = {win_loso_mean:.3f}")
ax.axhline(trl_loso_mean, color="darkorange",  linestyle="--", alpha=0.6,
           label=f"Mean Trial = {trl_loso_mean:.3f}")
ax.axhline(0.5, color="grey", linestyle=":", label="Chance (0.50)")

ax.set_xticks(x)
ax.set_xticklabels(sids, rotation=45, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("LOSO Cross-Subject Evaluation — Window & Trial Accuracy")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loso_accuracy.png", dpi=150)
plt.show()

# ── Save LOSO results ─────────────────────────────────────────────────────────
loso_out = {
    "per_subject": loso_results,
    "mean_window_accuracy": win_loso_mean,
    "mean_trial_accuracy":  trl_loso_mean,
}
with open(OUTPUT_DIR / "loso_results.json", "w") as fp:
    json.dump(loso_out, fp, indent=2)

print(f"\n✔  LOSO results saved to {OUTPUT_DIR / 'loso_results.json'}")
print("\n" + "=" * 55)
print("  FINAL SUMMARY")
print("=" * 55)
print(f"  Within-subject (test split)")
print(f"    Window accuracy : {window_acc:.4f}")
print(f"    Trial  accuracy : {trial_acc:.4f}")
print(f"  Cross-subject (LOSO)")
print(f"    Mean window acc : {win_loso_mean:.4f}")
print(f"    Mean trial  acc : {trl_loso_mean:.4f}")
print("=" * 55)